# this was ran in colab 

In [1]:
# 1. Install the missing dependency (zstd)
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Re-run the Ollama installation
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Install Python libraries
!pip install ollama datasets tqdm huggingface_hub python-dotenv

import subprocess
import time
import os

# 4. Start Ollama server
OLLAMA_PATH = "/usr/local/bin/ollama"

if os.path.exists(OLLAMA_PATH):
    print(f"Ollama installed successfully! Starting server...")
    # Start server in background
    subprocess.Popen([OLLAMA_PATH, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(15) # Wait for it to wake up

    # 5. Pull the model
    print("Pulling Llama 3.1 8B (this may take a minute)...")
    !{OLLAMA_PATH} pull llama3.1:8b
else:
    print("Ollama binary still not found. Check the installation logs above for errors.")

!ollama pull llama3.2:3b

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 129 kB in 1s (89.9 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (so

In [ ]:
import os
import json
import ollama
from tqdm import tqdm
from datasets import load_dataset, Dataset
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()
# --- CONFIGURATION ---
CHECKPOINT_FILE = "../../Data/json/dataset_checkpoint_ollama.jsonl"
os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)
REPO_ID = "mariaam22/business-qa-json-instruct-modified"
HF_TOKEN = os.getenv("HF_TOKEN")
MODEL_NAME = "llama3.2:3b"
PUSH_INTERVAL = 100

def clean_json_output(text):
    """Extracts JSON and removes any conversational fluff."""
    try:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1:
            json_str = text[start:end+1]
            return json.loads(json_str)
    except:
        return None
    return None

from datasets import concatenate_datasets, load_dataset

def push_to_hf(checkpoint_file, repo_id):
    """Safely APPENDS local progress to the existing Hugging Face dataset."""
    local_data = []
    
    # 1. Load the new local data
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    row = json.loads(line)
                    clean_row = {
                        "question": str(row.get("question", "")),
                        "context": str(row.get("context", "")),
                        "answer": row.get("answer") if isinstance(row.get("answer"), str) else json.dumps(row.get("answer"))
                    }
                    local_data.append(clean_row)
                except: continue
                
    if not local_data:
        return 0

    try:
        # 2. Convert local data to a Dataset object
        local_ds = Dataset.from_list(local_data)
        
        # 3. Try to pull the EXISTING dataset from Hugging Face
        try:
            print("\n[Cloud Sync] Pulling existing data from Hugging Face to merge...")
            existing_ds = load_dataset(repo_id, split="train", token=HF_TOKEN)
            
            # Combine the old data with the new local data
            combined_ds = concatenate_datasets([existing_ds, local_ds])
        except Exception as e:
            # If the dataset doesn't exist yet, just use the local data
            combined_ds = local_ds
            
        # 4. Push the combined data safely
        combined_ds.push_to_hub(repo_id, private=True, token=HF_TOKEN)
        
        # 5. Clear the local checkpoint file so we don't upload duplicates next time
        open(checkpoint_file, 'w').close() 
        
        return len(combined_ds)
        
    except Exception as e:
        tqdm.write(f"\n[!] HF Push Failed: {e}")
        return 0

def process_with_ollama(row):
    """
    Forces the AI to create unique, specific instructions and custom 
    JSON schemas for every single row.
    """
    prompt = f"""
    [SYSTEM]
    You are a high-diversity synthetic data generator. 
    Your goal is to create a "Challenge" for an AI model.
    
    [INPUT DATA]
    Context: {row['context']}
    Original Answer: {row['answer']}

    [TASK]
    Create a JSON object with two keys:
    1. "new_question": A highly specific instruction. DO NOT use generic phrases. 
       Instead of "Extract into JSON", use things like: 
       - "Create a technical profile for {row.get('company_name', 'this startup')} using a JSON schema that includes 'market_pain_point' and 'founder_pedigree'."
       - "Analyze the business failure described and output a JSON summary with keys for 'failure_mode' and 'post_mortem_insight'."
       - "Generate a business intelligence JSON object for this YC company focusing on its 'automation_vertical'."

    2. "new_answer": A structured JSON object that matches the specific keys you asked for in the question.

    [STRICT RULES]
    - The question MUST specify which keys to use.
    - The answer MUST be valid JSON.
    - No preamble or "Description:" text outside the JSON.
    """
    
    try:
        response = ollama.generate(
            model=MODEL_NAME,
            prompt=prompt,
            format='json',
            options={
                'temperature': 0.8, 
                'top_p': 0.9
            }
        )
        return clean_json_output(response['response'])
    except Exception as e:
        print(f"Error: {e}")
        return None

def main():
    login(token=HF_TOKEN)
    dataset = load_dataset("Dohahemdann/business-qa-analysis", split="train")
    
    # --- NEW LOGIC: Slice the dataset to start at START_ROW ---
    if START_ROW > 0 and START_ROW < len(dataset):
        print(f"Hard-skipping the first {START_ROW} rows of the dataset...")
        dataset = dataset.select(range(START_ROW, len(dataset)))
    
    processed_ids = set()
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed_ids.add(data["original_question"] + data["context"])
                except: continue
    
    remaining_rows = [row for row in dataset if (row['question'] + row['context']) not in processed_ids]
    print(f"Skipping {len(processed_ids)} already processed from checkpoint. Remaining to process: {len(remaining_rows)}")
    
    success_count = 0
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
        pbar = tqdm(remaining_rows)
        for row in pbar:
            result = process_with_ollama(row)
            
            if result and "new_question" in result and "new_answer" in result:
                output_row = {
                    "original_question": row['question'],
                    "question": result["new_question"],
                    "context": row['context'],
                    "answer": json.dumps(result["new_answer"], indent=2)
                }
                f.write(json.dumps(output_row) + "\n")
                f.flush() 
                
                success_count += 1
                pbar.set_description(f"Success: {success_count}")

                if success_count > 0 and success_count % PUSH_INTERVAL == 0:
                    push_to_hf(CHECKPOINT_FILE, REPO_ID)

    push_to_hf(CHECKPOINT_FILE, REPO_ID)

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Skipping 0. Remaining: 11565


Success: 100:   1%|          | 102/11565 [12:42<28:48:57,  9.05s/it]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  70%|#######   |  577kB /  824kB            

README.md:   0%|          | 0.00/348 [00:00<?, ?B/s]

Success: 1355:  12%|█▏        | 1400/11565 [4:05:14<19:16:02,  6.82s/it]

In [5]:
# import json
# import os
# from datasets import Dataset
# from huggingface_hub import login

# # --- SETTINGS ---
# CHECKPOINT_FILE = "dataset_checkpoint_ollama.jsonl"
# # REPO_ID = "mariaam22/business-qa-json-instruct"
# # HF_TOKEN = "your_hf_token_here" # Paste your token here

# def manual_push_to_hub():
#     # 1. Login
#     login(token=HF_TOKEN)

#     # 2. Read the file
#     final_data = []
#     if not os.path.exists(CHECKPOINT_FILE):
#         print(f"Error: {CHECKPOINT_FILE} not found!")
#         return

#     print(f"Reading data from {CHECKPOINT_FILE}...")
#     with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
#         for line in f:
#             try:
#                 row = json.loads(line)

#                 # CLEANING: Ensure all fields are strings to avoid API errors
#                 # We extract the 'question', 'context', and 'answer'
#                 clean_row = {
#                     "question": str(row.get("question", "")),
#                     "context": str(row.get("context", "")),
#                     # If answer is a dict, convert to string; otherwise keep as string
#                     "answer": row.get("answer") if isinstance(row.get("answer"), str) else json.dumps(row.get("answer"))
#                 }
#                 final_data.append(clean_row)
#             except Exception as e:
#                 continue

#     # 3. Push to Hub
#     if final_data:
#         print(f"Found {len(final_data)} rows. Creating Dataset...")
#         new_ds = Dataset.from_list(final_data)

#         print(f"Pushing to {REPO_ID}...")
#         new_ds.push_to_hub(REPO_ID, private=True)
#         print("Done! Check your Hugging Face profile.")
#     else:
#         print("No valid data found in the file.")

# if __name__ == "__main__":
#     manual_push_to_hub()

Reading data from dataset_checkpoint_ollama.jsonl...
Found 107 rows. Creating Dataset...
Pushing to mariaam22/business-qa-json-instruct...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 73.6kB / 73.6kB            

Done! Check your Hugging Face profile.
